## 01-Classical Machine Learning Baseline
In this notebook, we will do a simple baseline classification and regression using classical methods (Tfidf, logistic regression, ridge)

In [24]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    mean_squared_error,
)
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.pipeline import make_pipeline

In [25]:
train_fname = "../data/splits/train.csv"
dev_fname = "../data/splits/dev.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 132
Dev df = 34


In [26]:
train_df.head()

,text,target,mmse
0,Reaching for a cookie chair possibly moving to...,CN,28
1,As a picture little girl and a little boy stan...,AD,18
2,Tell me everything that you see happening. Eve...,CN,30
3,I think what I want you to do is take a look a...,CN,30
4,Tell me all the action that you can say on wha...,AD,20


In [27]:
dev_df.head()

,text,target,mmse
0,"Everything you see going on in this picture, g...",CN,28
1,"Now you just look at the picture. Okay, this i...",CN,30
2,Tell me everything that you see going on in th...,AD,20
3,Girls and boys getting her one that stores sto...,AD,20
4,"All right, we have a few more tasks to go thro...",CN,30


### Pre-processing
We will lowercase the text for better efficiency

In [28]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

### Build Tfidf Vectorizer

We can use Tfidf Vectorizer to convert text to a fixed-length vector

In [29]:
# include ngrams
vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 3))

### Train Logistic regression using Tfidf vectorizer

We can now train a simple logistic regression classifier using processed X_train

In [30]:
print("Training logistic classifier pipeline...")
pipeline = make_pipeline(
    vectorizer, LogisticRegressionCV(cv=10, max_iter=200, random_state=42)
)
pipeline.fit(train_df["text_lower"], train_df["target"])

print("Completed")
y_pred = pipeline.predict(train_df["text_lower"])
print(classification_report(train_df["target"], y_pred, target_names=["AD", "CN"]))

Training logistic classifier pipeline...
Completed
              precision    recall  f1-score   support

          AD       1.00      0.95      0.98        66
          CN       0.96      1.00      0.98        66

    accuracy                           0.98       132
   macro avg       0.98      0.98      0.98       132
weighted avg       0.98      0.98      0.98       132



### Now we can evaluate on dev set

In [31]:
y_pred = pipeline.predict(dev_df["text_lower"])
print(classification_report(dev_df["target"], y_pred, target_names=["AD", "CN"]))

              precision    recall  f1-score   support

          AD       0.81      0.81      0.81        21
          CN       0.69      0.69      0.69        13

    accuracy                           0.76        34
   macro avg       0.75      0.75      0.75        34
weighted avg       0.76      0.76      0.76        34



The baseline result is reasonable for dev set (76% precision, 76% recall). 
However, it's much lower than the scores for train set (98% for precision, 98% for recall). It means our model is likely to overfit on train data. 

### Train Ridge using Tfidf vectorizer

We can now train a ridge regression model using processed X_train

In [32]:
print("Training ridge pipeline...")
pipeline = make_pipeline(vectorizer, RidgeCV(cv=10))
pipeline.fit(train_df["text_lower"], train_df["mmse"])
print("Completed")
y_pred = pipeline.predict(train_df["text_lower"])
print("Training RMSE: ", np.sqrt(mean_squared_error(train_df["mmse"], y_pred)))

Training ridge pipeline...
Completed
Training RMSE:  0.6383232901273279


In [33]:
y_pred = pipeline.predict(dev_df["text_lower"])
print("Dev RMSE: ", np.sqrt(mean_squared_error(dev_df["mmse"], y_pred)))

Dev RMSE:  5.688216067990933


Again, we see that the cross-validated ridge model is also clearly overfitting in this case.